In [39]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
from numpy import random
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable
from glob import glob

In [ ]:
sys.path.append(os.getcwd()+'/Source')

from OP_Kernels import *
from OP_functions import *

: 

In [ ]:
def load_config(config_yaml):
   with open(config_yaml) as f:
       config = yaml.safe_load(f)
   return config

: 

In [ ]:
param = load_config('yaml/config.yaml')
start = datetime(param['startdate']['year'], param['startdate']['month'], param['startdate']['day']) #Start date
length = 1#param['param']['length'] # Set Time length [days] 
dt = param['param']['dt'] #toggle between - or + to pick backwards or forwards 
N = param['param']['N'] # Name of deploy loc file
dd = param['param']['dd'] #max depth difference z in N
name = param['file']['name'] #name output file
dtp = param['param']['dtp'] #how often particle released in hours
odt = param['param']['odt'] #how often data is recorded
rrr = param['param']['r'] #radious of particle deployment
SI = param['param']['SI'] #how many particles per super-individual

duration = timedelta(days=length)
print(f"The model will run for {duration.days} days, starting at {start}")

lon, lat, z = pandas_deploy(N,SI,rrr,dd,dtp)
N = len(lat)
print(f"The model will release {N} particle every timestep")

daterange = [start+timedelta(days=i) for i in range(length)]
fn =  name + '.zarr'
outfile = os.path.join(os.getcwd()+'/results/', fn)

print(f"Output file: {outfile}")

The model will run for 1 days, starting at 2007-04-01 00:00:00
The model will release 3396 particle every timestep
Output file: /ocean/jvalenti/MOAD/analysis-jose/notebooks/MP_model/results/test_reseed.zarr


: 

In [ ]:
chunksize_fields = {
    "lon": ("lon", 66),   # matches 1 MPI box in lon
    "lat": ("lat", 28),   # matches 1 MPI box in lat
    "time": ("time", 1)
}

: 

In [40]:
def filename_set(
    start,
    length,
    varlist=['U', 'V', 'W'],
    machine='nibi'
):
    duration = timedelta(days=length)
    paths = path(machine)

    Tlist, Ulist, Vlist, Wlist = [], [], [], []
    Waveslist = []

    for day in range(duration.days):
        path_NEMO = make_prefix(
            start + timedelta(days=day),
            paths['NEMO']
        )
        print(path_NEMO)

        path_NEMO_d = make_prefix(
            start + timedelta(days=day),
            paths['NEMO'],
            res='d'
        )

        Ulist.append(path_NEMO + '_grid_U.nc')
        Vlist.append(path_NEMO + '_grid_V.nc')
        Wlist.append(path_NEMO + '_grid_W.nc')
        Tlist.append(path_NEMO + '_grid_T.nc')

    # ------------------------------------------------------------------
    # Filenames
    # ------------------------------------------------------------------
    filenames = {
        # NEMO C-grid velocity fields
        'U': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Ulist[0],
            'data': Ulist
        },

        'V': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Vlist[0],
            'data': Vlist
        },

        'W': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Wlist[0],
            'data': Wlist
        },

        # Time-varying vertical coordinate
        'depth_w': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Wlist[0],
            'data': Wlist
        },

        # Other fields
        'Kz': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Wlist[0],
            'data': Wlist
        },

        'T': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Tlist[0],
            'data': Tlist
        },

        'S': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Tlist[0],
            'data': Tlist
        },

        'ssh': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'data': Tlist
        },

        'R': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Tlist[0],
            'data': Tlist
        },

        'Bathy': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'data': paths['bat']
        },

        'gdepth': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'depth': Wlist[0],
            'data': paths['mask']
        },

        'totdepth': {
            'lon': paths['coords'],
            'lat': paths['coords'],
            'data': paths['mask']
        },

        'US': {
            'lon': paths['coordsWW3'],
            'lat': paths['coordsWW3'],
            'data': Waveslist
        },

        'VS': {
            'lon': paths['coordsWW3'],
            'lat': paths['coordsWW3'],
            'data': Waveslist
        },

        'WL': {
            'lon': paths['coordsWW3'],
            'lat': paths['coordsWW3'],
            'data': Waveslist
        }
    }

    # ------------------------------------------------------------------
    # NetCDF variable names
    # ------------------------------------------------------------------
    variables = {
        'U': 'vozocrtx',
        'V': 'vomecrty',
        'W': 'vovecrtz',
        'depth_w': 'depthw',
        'depth_u': 'depthu',

        'Kz': 'vert_eddy_diff',
        'T': 'votemper',
        'S': 'vosaline',
        'R': 'sigma_theta',

        'US': 'uuss',
        'VS': 'vuss',
        'WL': 'lm',

        'Bathy': 'Bathymetry',
        'ssh': 'sossheig',
        'totdepth': 'totaldepth'
    }

    # ------------------------------------------------------------------
    # NetCDF dimensions
    # ------------------------------------------------------------------
    dimensions = {
        # NEMO C-grid velocity fields
        'U': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'not_yet_set',
            'time': 'time_counter'
        },

        'V': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'not_yet_set',
            'time': 'time_counter'
        },

        'W': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'not_yet_set',
            'time': 'time_counter'
        },

        # Time-varying W-point depth
        'depth_w': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'not_yet_set',
            'time': 'time_counter'
        },
        
        'depth_u': {
                    'lon': 'glamf',
                    'lat': 'gphif',
                    'depth': 'not_yet_set',
                    'time': 'time_counter'
                },

        # Kz is also on the vertical W grid
        'Kz': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'depthw',
            'time': 'time_counter'
        },

        # Tracers
        'T': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'deptht',
            'time': 'time_counter'
        },

        'S': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'deptht',
            'time': 'time_counter'
        },

        'R': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'deptht',
            'time': 'time_counter'
        },

        # 2D fields
        'ssh': {
            'lon': 'glamf',
            'lat': 'gphif',
            'time': 'time_counter'
        },

        'Bathy': {
            'lon': 'glamf',
            'lat': 'gphif'
        },

        'gdepth': {
            'lon': 'glamf',
            'lat': 'gphif',
            'depth': 'depthw'
        },

        'totdepth': {
            'lon': 'glamf',
            'lat': 'gphif'
        },

        # Wave fields
        'US': {
            'lon': 'lon',
            'lat': 'lat',
            'time': 'time'
        },

        'VS': {
            'lon': 'lon',
            'lat': 'lat',
            'time': 'time'
        },

        'WL': {
            'lon': 'lon',
            'lat': 'lat',
            'time': 'time'
        }
    }

    # ------------------------------------------------------------------
    # Select requested variables
    # ------------------------------------------------------------------
    file_out = {}
    var_out = {}
    dim_out = {}

    for var in varlist:
        file_out[var] = filenames[var]
        var_out[var] = variables[var]
        dim_out[var] = dimensions[var]

    return file_out, var_out, dim_out

In [49]:
import xarray as xr

xr.open_dataset('/results2/SalishSea/nowcast-green.202111/01apr07/SalishSea_1h_20070401_20070401_grid_T.nc')

<xarray.Dataset> Size: 6GB
Dimensions:               (y: 898, x: 398, nvertex: 4, deptht: 40,
                           axis_nbounds: 2, time_counter: 24)
Coordinates:
    nav_lat               (y, x) float32 1MB ...
    nav_lon               (y, x) float32 1MB ...
  * deptht                (deptht) float32 160B 0.5 1.5 2.5 ... 414.5 441.5
    time_centered         (time_counter) datetime64[ns] 192B ...
  * time_counter          (time_counter) datetime64[ns] 192B 2007-04-01T00:30...
Dimensions without coordinates: y, x, nvertex, axis_nbounds
Data variables:
    bounds_lon            (y, x, nvertex) float32 6MB ...
    bounds_lat            (y, x, nvertex) float32 6MB ...
    area                  (y, x) float32 1MB ...
    deptht_bounds         (deptht, axis_nbounds) float32 320B ...
    sossheig              (time_counter, y, x) float32 34MB ...
    time_centered_bounds  (time_counter, axis_nbounds) datetime64[ns] 384B ...
    time_counter_bounds   (time_counter, axis_nbounds) datetime64[ns] 384B ...
    votemper              (time_counter, deptht, y, x) float32 1GB ...
    vosaline              (time_counter, deptht, y, x) float32 1GB ...
    sigma_theta           (time_counter, deptht, y, x) float32 1GB ...
    e3t                   (time_counter, deptht, y, x) float32 1GB ...
Attributes:
    name:         SalishSea_1h_20070401_20070405
    description:  physics tracers and VVL layer thicknesses
    title:        physics tracers and VVL layer thicknesses
    Conventions:  CF-1.6
    timeStamp:    2022-Oct-25 15:59:54 GMT
    uuid:         e748ee8f-c4d3-4506-a3b2-cbc3f327747d

In [41]:
filenames, variables, dimensions = filename_set(
    start,
    length,
    varlist=['U', 'V', 'W','depth_w']
,machine='salish')

field_set = FieldSet.from_netcdf(
    filenames,
    variables,
    dimensions,
    mesh='flat',
    allow_time_extrapolation=True
)

/results2/SalishSea/nowcast-green.202111/01apr07/SalishSea_1h_20070401_20070401


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might b

IndexError: tuple index out of range

In [37]:
depth_w = Field.from_netcdf(
    filenames['depth_w'],
    ('depth_w', variables['depth_w']),
    dimensions['depth_w'],
    mesh='flat'
)

KeyError: 'depth_w'

In [29]:
filenames, variables, dimensions = filename_set(
    start,
    length,
    varlist=[ 'depth_w']
,machine='salish')

field_set = FieldSet.from_netcdf(
    filenames,
    variables,
    dimensions,
    mesh='flat',
    allow_time_extrapolation=True
)

field_set.U.set_depth_from_field(field_set.depth_u)
field_set.V.set_depth_from_field(field_set.depth_u)
field_set.W.set_depth_from_field(field_set.depth_w)

/results2/SalishSea/nowcast-green.202111/01apr07/SalishSea_1h_20070401_20070401


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(


IndexError: tuple index out of range

In [16]:
####BUILD FIELDS FOR SIMULATION######
#Fill in the list of variables that you want to use as fields
varlist=['U','V','W']
filenames,variables=filename_set(start,length,varlist,machine='salish')
dimensions = {'lon': 'glamf', 'lat': 'gphif', 'depth': 'depthw','time': 'time_counter'}
field_set=FieldSet.from_nemo(filenames, variables, dimensions, allow_time_extrapolation=True)

#Find file names and variable names ###'Diat','Flag'###
varlist=['US','VS','WL','R','T','S','ssh','Bathy','Kz','totdepth']
filenames,variables=filename_set(start,length,varlist,machine='salish')

# #Add Stokes Drift fields
# dimensions = {'lon': 'longitude', 'lat': 'latitude', 'time': 'time'}
# us = Field.from_netcdf(filenames['US'], variables['US'], dimensions,allow_time_extrapolation=True)
# vs = Field.from_netcdf(filenames['VS'], variables['VS'], dimensions,allow_time_extrapolation=True)
# wl = Field.from_netcdf(filenames['WL'], variables['WL'], dimensions,allow_time_extrapolation=True)
# field_set.add_field(us)
# field_set.add_field(vs)
# field_set.add_field(wl)
# field_set.add_vector_field(VectorField("stokes", us, vs, wl))

#Add Vertical diffusivity coefficient field
dimensions = {'lon': 'glamt', 'lat': 'gphit', 'depth': 'depthw','time': 'time_counter'}
Kz = Field.from_netcdf(filenames['Kz'], variables['Kz'], dimensions,allow_time_extrapolation=True) 
field_set.add_field(Kz)

#Add fields located at node T
dimensions = {'lon': 'glamt', 'lat': 'gphit', 'depth': 'deptht','time': 'time_counter'}

R = Field.from_netcdf(filenames['R'], variables['R'], dimensions,allow_time_extrapolation=True)
S = Field.from_netcdf(filenames['S'], variables['S'], dimensions,allow_time_extrapolation=True)
T = Field.from_netcdf(filenames['T'], variables['T'], dimensions,allow_time_extrapolation=True)
field_set.add_field(R)
field_set.add_field(S)
field_set.add_field(T)

#Add Bathymetry 2D field
dimensions = {'lon': 'glamt', 'lat': 'gphit'}
Bth = Field.from_netcdf(filenames['Bathy'], variables['Bathy'], dimensions,allow_time_extrapolation=True)
TD = Field.from_netcdf(filenames['totdepth'], variables['totdepth'], dimensions,allow_time_extrapolation=True)
field_set.add_field(Bth)
field_set.add_field(TD)

#Add SSH 
dimensions = {'lon': 'glamt', 'lat': 'gphit','time': 'time_counter'}
SSH = Field.from_netcdf(filenames['ssh'], variables['ssh'], dimensions,allow_time_extrapolation=True)
field_set.add_field(SSH)

/results2/SalishSea/nowcast-green.202111/01apr07/SalishSea_1h_20070401_20070401
/results2/SalishSea/nowcast-green.202111/02apr07/SalishSea_1h_20070402_20070402
/results2/SalishSea/nowcast-green.202111/03apr07/SalishSea_1h_20070403_20070403
/results2/SalishSea/nowcast-green.202111/04apr07/SalishSea_1h_20070404_20070404
/results2/SalishSea/nowcast-green.202111/05apr07/SalishSea_1h_20070405_20070405


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might b

/results2/SalishSea/nowcast-green.202111/01apr07/SalishSea_1h_20070401_20070401
/results2/SalishSea/nowcast-green.202111/02apr07/SalishSea_1h_20070402_20070402
/results2/SalishSea/nowcast-green.202111/03apr07/SalishSea_1h_20070403_20070403
/results2/SalishSea/nowcast-green.202111/04apr07/SalishSea_1h_20070404_20070404
/results2/SalishSea/nowcast-green.202111/05apr07/SalishSea_1h_20070405_20070405


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might b

In [8]:
#MPI decomposition
Px = 6  # number of boxes along x (longitude)
Py = 32   # number of boxes along y (latitude)
mpi_size = Px * Py  # should equal 192 NIBI

# Set domain bounds from the model grid
xmin = field_set.U.grid.lon.min()
xmax = field_set.U.grid.lon.max()
ymin = field_set.U.grid.lat.min()
ymax = field_set.U.grid.lat.max()

x_edges = np.linspace(xmin, xmax, Px + 1)
y_edges = np.linspace(ymin, ymax, Py + 1)

def nemo_partition_function(pset, mpi_size): 
    """Custom partition function for Parcels similar to NEMO MPI decomposition.
    Returns an array of processor IDs for each particle."""
    
    lon = pset.lon
    lat = pset.lat

    # Find which box each particle belongs to
    ix = np.searchsorted(x_edges, lon) - 1
    iy = np.searchsorted(y_edges, lat) - 1

    # Clamp indices to valid range
    ix = np.clip(ix, 0, Px - 1)
    iy = np.clip(iy, 0, Py - 1)

    # Map 2D box index to processor ID
    proc_id = iy * Px + ix

    return proc_id

In [9]:
#Define our Parcels Particles
class MPParticle(JITParticle):    
    diameter = Variable('diameter', initial = param['particle']['diameter'])
    length = Variable('length', initial = param['particle']['length'])
    Ub = Variable('Ub', initial = param['particle']['Ub'])  
    status = Variable('status', initial = 0)
    vvl_factor = Variable('fact', initial =  1)    
    ws = Variable('ws', initial =  0) 
    wa = Variable('wa', initial =  0) 
    wm = Variable('wm', initial =  0) 
    alpha = Variable('alpha',initial=param['particle']['alpha'])

In [10]:
pset = ParticleSet.from_list(field_set, MPParticle, lon=lon, lat=lat, depth=z, repeatdt = timedelta(hours=dtp))#,partition_function=nemo_partition_function)

In [ ]:
pset.execute([Advection,Buoyancy,turb_mix,Displacement,export,CheckOutOfBounds,KeepInOcean],
        runtime=duration, 
        dt=dt,
        output_file=pset.ParticleFile(name=outfile, outputdt=timedelta(hours=odt)))#,chunks=(int(1e5), 100)))

INFO: Output files are stored in /ocean/jvalenti/MOAD/analysis-jose/notebooks/MP_model/results/test_reseed.zarr.
